# Day 5.6 — MCP Client: Discover, Then Govern

Model Context Protocol is a standard way for a client to ask a server *what can you do?*
and then *do this*. It is genuinely useful: one client can talk to tools nobody on your
team wrote.

It also grants exactly no authority. The server describes itself; your harness decides
what that description is worth.


## Before you begin

### Learning outcomes

- Discover a tool over an MCP-shaped client and read its schema.
- Classify a discovered tool locally and run it past your own policy.
- Connect to a real stdio MCP server, or degrade cleanly when the SDK is absent.

Architecture reference: [Day 5 diagrams D17](../../diagrams/source/day_05.md).

### Expected observation

`course_lookup` is discovered, classified `read`, allowed, and called. Re-classifying the same tool as `external` changes the decision while discovery is unchanged.


## Concept briefing

## MCP: protocol, not permission

Model Context Protocol lets a client initialise a session, discover server capabilities
and invoke them through a common contract. A server may expose tools, resources or prompts.
The protocol improves interoperability; it does not establish trust.

An MCP tool description and its results are untrusted external content. Before importing
a discovered tool, the harness should consider server origin, schema, local risk,
permitted agents, arguments, timeout, output handling and logging. A server changing its
advertised tools must not silently expand application authority.

The Day 5 rule is therefore:

```text
discovery is not authorization
```

The client discovers the tool, the harness classifies it, local policy authorises or
pauses it, and only then does the protocol call occur.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — Discovery, offline

`FakeMCPClient` speaks the same shape as the real SDK: a list of tool objects with
`.name`, `.description` and `.inputSchema`. Learning one access pattern is the point.


In [ ]:
from mini_harness import FakeMCPClient

client = FakeMCPClient()
tools = await client.list_tools()          # notebooks allow top-level await

print("Tools advertised by the server:", len(tools))
for tool in tools:
    print("  object type :", type(tool).__name__)
    print("  .name       :", tool.name)
    print("  .description:", tool.description)
    print("  .inputSchema:", tool.inputSchema)

print()
print("Everything above is text the SERVER chose to send us. It is evidence, not truth.")

## Step 2 — Classify it yourself before it enters the harness

The server does not get to say how risky its own tool is. We convert the description into
a local `ToolSpec` and assign the risk level ourselves.


In [ ]:
from mini_harness import AgentConfig, ToolSpec, decide

remote = tools[0]
spec = ToolSpec(
    name=remote.name,
    description=remote.description,
    input_schema=remote.inputSchema,
    risk="read",                    # OUR judgement: it only returns a fact
)
mcp_agent = AgentConfig("mcp_demo", "Use one supplied fact.", allowed_tools=[spec.name])

print("Imported as   :", spec)
print("Local risk    :", spec.risk, "(assigned by us, not by the server)")
print("Policy decision:", decide(mcp_agent, spec))

## Step 3 — Only now do we call it

Note the order: discover, classify, authorise, *then* invoke. `tool_result_payload`
normalises the answer so the fake and a real server are read the same way.


In [ ]:
from mini_harness import tool_result_payload

if decide(mcp_agent, spec) == "allow":
    raw = await client.call_tool(spec.name, {"topic": "mcp"})
    print("Raw result object:", type(raw).__name__)
    print("Normalised payload:", tool_result_payload(raw))
else:
    print("Policy refused; no protocol call was made.")

## Step 4 — Change the classification, not the server

The server is untouched. The description is identical. Only our local risk label moved,
and the decision moved with it.


In [ ]:
reclassified = ToolSpec(remote.name, remote.description, remote.inputSchema, "external")

print("Discovery is unchanged:", reclassified.name == spec.name,
      "| same schema:", reclassified.input_schema == spec.input_schema)
print()
print("risk = read     -> decision:", decide(mcp_agent, spec))
print("risk = external -> decision:", decide(mcp_agent, reclassified))
print()
print("This is 'discovery is not authorization' in one comparison.")
print("A server that renames a tool, or quietly changes what it does, cannot")
print("expand what your agent is permitted to do - because it never set the risk.")

## Step 5 — A real MCP server over stdio

`instructor_mcp_server.py` is a genuine MCP server. This cell launches it as a subprocess,
completes a protocol session and calls one tool. Two details that matter on Windows: the
command must be `sys.executable` (the interpreter running this notebook, never the literal
string `"python"`), and the session runs in `..._sync`, which gives the subprocess its own
event loop. If the SDK is not installed, the cell says how to install it and moves on.


In [ ]:
import importlib.util

MCP_AVAILABLE = importlib.util.find_spec("mcp") is not None
print("MCP SDK installed:", MCP_AVAILABLE)

if not MCP_AVAILABLE:
    print('Optional: pip install "mcp>=1.27,<2" to run this cell against a real server.')
    print("Everything above already ran against FakeMCPClient, so nothing is missing.")
else:
    from mini_harness import StdioMCPClient
    server = PROJECT_ROOT / "instructor_mcp_server.py"
    real = StdioMCPClient(sys.executable, [str(server)])   # sys.executable, not "python"
    try:
        real_tools, real_result = real.list_and_optionally_call_sync(
            "course_lookup", {"topic": "harness"})
        print("Server:", server.name)
        for tool in real_tools:
            print("  object type :", type(tool).__name__, "(from the real SDK)")
            print("  .name       :", tool.name)
            print("  .description:", tool.description)
            print("  .inputSchema:", tool.inputSchema)
        print("Result payload:", tool_result_payload(real_result))
        print()
        print("Same three attributes as the fake, read with the same code.")
        print("That is why Step 1 was worth doing offline first.")
    except Exception as exc:                 # noqa: BLE001 - never stop the class
        print("Real MCP session failed:", type(exc).__name__, exc)
        print("Use the FakeMCPClient path above; the lesson is unchanged.")

Note what did **not** change. The real server returned real objects, and the
policy step in Step 2 would be identical: classify it locally, then ask `decide`.
Nothing about "this came from a real server" makes it more trusted.


### Try it yourself

Predict what happens when you ask an MCP server for a tool it never advertised.


In [ ]:
# --- Worked solution ---
# Two separate protections, and they fail at different moments.

# 1. Asking the client directly for an unknown tool: the server refuses.
try:
    await client.call_tool("delete_everything", {})
except KeyError as exc:
    print("Server side  : unknown tool ->", type(exc).__name__, exc)

# 2. The more important case: the tool IS advertised, but is not in our allow-list.
sneaky = ToolSpec("delete_everything", "Advertised by the server", {"type": "object"}, "read")
print("Advertised by the server, but on our allow-list?",
      sneaky.name in mcp_agent.allowed_tools)
print("Local policy decision:", decide(mcp_agent, sneaky))
print()
print("Even labelled 'read', it is denied - because policy checks the allow-list first.")
print("A server can advertise anything it likes; it cannot add itself to your allow-list.")

### Checkpoint

**1. An MCP server updates and now advertises a `publish_site` tool. What can it do?**

<details><summary>Show answer</summary>

Nothing, on its own. Discovery only tells your harness the tool exists. It is not in any agent's `allowed_tools`, and nobody has given it a local risk level, so `decide` returns `deny`. Someone has to make a deliberate change to your configuration before it can run.

</details>

**2. Why does the fake client return objects rather than plain dictionaries?**

<details><summary>Show answer</summary>

Because the real SDK returns objects with `.name`, `.description` and `.inputSchema`. If the fake returned dictionaries you would learn `tool["name"]`, and every line of that code would break the first time you pointed it at a real server.

</details>

### Recap

- Limitation: a protocol makes outside capabilities reachable, which is exactly what makes it dangerous - the server writes its own description.
- Layer added: local classification of every discovered tool into a `ToolSpec`, checked by the same `policy.decide` used for local tools.
- Evidence: one tool discovered and called only after being allowed; re-labelling it `external` changed the decision without touching the server; an advertised but un-allow-listed tool was denied.
